<a href="https://colab.research.google.com/github/Nakul-Narang/Qwen_fine_tune/blob/main/Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to fine-tune a pre-trained large language model (LLM) using the QLoRA technique on a custom dataset. The process involves installing necessary libraries, preparing the dataset, configuring quantization, setting up LoRA, and finally training and testing the fine-tuned model.

## Step 1: Install Libraries

This section installs the required Python libraries using `pip`. These libraries are essential for working with Hugging Face models (`transformers`, `datasets`), efficient low-rank adaptation (`peft`, `trl`), accelerating training (`accelerate`), and 4-bit quantization (`bitsandbytes`).

## Step 2: Imports

Here, all necessary modules and classes from the installed libraries are imported. This includes `torch` for tensor operations, `Dataset` for creating datasets, various components from `transformers` for model loading and training arguments, `LoraConfig` and `get_peft_model` from `peft` for QLoRA, and `SFTTrainer` from `trl` for supervised fine-tuning.

## Step 3: Check GPU

This step verifies the availability of a GPU and displays its name and available VRAM. A GPU is crucial for efficient training of large language models. The `torch.cuda` module is used for these checks.

## Step 4: Create Dataset

A small custom dataset is created for demonstration. It consists of medical question-response pairs. The data is first defined as a list of dictionaries, then converted into a `datasets.Dataset` object. A `format_example` function is used to structure each example into a specific prompt-response format suitable for instruction-tuned models, making it ready for fine-tuning.

## Step 5: Load Tokenizer

The `AutoTokenizer` class from `transformers` is used to load the tokenizer corresponding to the pre-trained model (`Qwen/Qwen2.5-1.5B-Instruct`). The tokenizer is responsible for converting text into numerical tokens that the model can understand. The `pad_token` is set to `eos_token` (end-of-sequence token) which is a common practice for causal language models during fine-tuning.

## Step 6: Configure 4-bit Quantization

`BitsAndBytesConfig` is used to set up 4-bit quantization. This technique reduces the memory footprint of the model, allowing larger models to be fine-tuned on less powerful GPUs. Key parameters include `load_in_4bit=True`, `bnb_4bit_quant_type="nf4"` (NF4 quantization), and `bnb_4bit_compute_dtype=torch.float16` for computation in float16 precision.

## Step 7: Load Quantized Model

The `AutoModelForCausalLM` is used to load the pre-trained `Qwen/Qwen2.5-1.5B-Instruct` model. The `quantization_config` is passed to load the model directly in 4-bit quantized format. `device_map="auto"` automatically distributes the model layers across available devices (e.g., GPU).

## Step 8: Prepare for QLoRA

`prepare_model_for_kbit_training` from `peft` is called. This function prepares the 4-bit quantized model for QLoRA training by, for example, casting certain modules to `float32` for stability and adding adapters where necessary. This step is crucial for enabling efficient LoRA fine-tuning on a quantized model.

## Step 9: LoRA Configuration

`LoraConfig` defines the parameters for the LoRA adapters. These parameters include `r` (LoRA attention dimension), `lora_alpha` (scaling factor), `lora_dropout`, `bias` (whether to fine-tune bias terms), `task_type` (`CAUSAL_LM` for generative models), and `target_modules` (the specific linear layers in the model where LoRA adapters will be applied, typically attention query, key, value, and output projections).

## Step 10: Attach LoRA

The `get_peft_model` function applies the `LoraConfig` to the prepared model, effectively attaching the LoRA adapters. After this step, only the small LoRA adapters and potentially a few other parameters are trainable, significantly reducing the number of parameters to update during fine-tuning. `model.print_trainable_parameters()` displays the percentage of trainable parameters, which should be very low.

## Step 11: Training Arguments

`TrainingArguments` specifies various parameters for the training process, such as the `output_dir` for saving checkpoints, `num_train_epochs`, `per_device_train_batch_size`, `gradient_accumulation_steps`, `learning_rate`, `logging_steps`, `save_strategy`, and `fp16` (set to `False` in the current configuration, meaning no mixed-precision training is used here). `report_to="none"` disables external reporting tools.

## Step 12: Trainer

The `SFTTrainer` (Supervised Fine-Tuning Trainer) from `trl` is initialized. It orchestrates the fine-tuning process, taking the LoRA-adapted model, the prepared training dataset, and the `TrainingArguments` as input. The `SFTTrainer` is designed for easily fine-tuning models on instruction datasets.

## Step 13: Train

`trainer.train()` starts the actual fine-tuning process. During this phase, the model's LoRA adapters are updated based on the training data and the specified training arguments. The progress and metrics are logged as configured in `TrainingArguments`.

## Step 14: Save Adapter

After training, the fine-tuned LoRA adapters (and the tokenizer configuration) are saved to a directory named `ecg_lora_adapter`. This saves only the small adapter weights, which can later be merged with the base model without saving the entire large model.

## Step 15: Test Inference

This section demonstrates how to use the fine-tuned model for inference. A `prompt` is created, tokenized using the saved tokenizer, and then passed to the model's `generate` method. The `max_new_tokens` parameter limits the length of the generated response. Finally, the generated tokens are decoded back into human-readable text and printed.

## Step 16: Zip Adapter

This command zips the saved adapter directory (`ecg_lora_adapter`) into a single `ecg_adapter.zip` file. This is useful for easily sharing or storing the fine-tuned adapters.

In [ ]:
# ==========================================
# STEP 1: Install Libraries
# ==========================================
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes

# ==========================================
# STEP 2: Imports
# ==========================================
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

# ==========================================
# STEP 3: Check GPU
# ==========================================
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

# ==========================================
# STEP 4: Create Dataset
# ==========================================
data = [
    {
        "instruction": "What is ECG?",
        "response": "ECG stands for Electrocardiogram and records the electrical activity of the heart."
    },
    {
        "instruction": "What is Arrhythmia?",
        "response": "Arrhythmia is an abnormal heart rhythm."
    },
    {
        "instruction": "What is Tachycardia?",
        "response": "Tachycardia is a condition where the heart beats faster than normal."
    }
]

dataset = Dataset.from_list(data)

def format_example(example):
    return {
        "text":
        f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

dataset = dataset.map(format_example)

# ==========================================
# STEP 5: Load Tokenizer
# ==========================================
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# STEP 6: Configure 4-bit Quantization
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ==========================================
# STEP 7: Load Quantized Model
# ==========================================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# ==========================================
# STEP 8: Prepare for QLoRA
# ==========================================
model = prepare_model_for_kbit_training(model)

# ==========================================
# STEP 9: LoRA Configuration
# ==========================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

# ==========================================
# STEP 10: Attach LoRA
# ==========================================
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

# ==========================================
# STEP 11: Training Arguments
# ==========================================
training_args = TrainingArguments(
    output_dir="outputs",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=False,
    report_to="none"
)

# ==========================================
# STEP 12: Trainer
# ==========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

# ==========================================
# STEP 13: Train
# ==========================================
trainer.train()

# ==========================================
# STEP 14: Save Adapter
# ==========================================
model.save_pretrained("ecg_lora_adapter")
tokenizer.save_pretrained("ecg_lora_adapter")

print("\nAdapter Saved Successfully!")

# ==========================================
# STEP 15: Test Inference
# ==========================================
prompt = "What is ECG?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print("\nModel Output:\n")
print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

# ==========================================
# STEP 16: Zip Adapter
# ==========================================
!zip -r ecg_adapter.zip ecg_lora_adapter

print("\nZIP File Created: ecg_adapter.zip")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.6 MB/s eta 0:00:00
GPU: Tesla T4
VRAM: 14.56 GB


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


...

Step,Training Loss
1,2.788475
2,2.271158
3,2.922251
4,2.722992
5,2.055638
6,1.734329
7,2.262176
8,1.501181
9,1.547665
10,1.206440


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Adapter Saved Successfully!


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Model Output:

What is ECG? What Ly Ly下面是小下面是小 Ly下面是小无单元 Ly Ly Ly下面是小 Ly Ly Ly Ly下面是小下面是小印翻 Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小 Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小印印印印下面是小 Ly Ly Ly下面是小下面是小下面是小下面是小
  adding: ecg_lora_adapter/ (stored 0%)
  adding: ecg_lora_adapter/chat_template.jinja (deflated 71%)
  adding: ecg_lora_adapter/tokenizer_config.json (deflated 60%)
  adding: ecg_lora_adapter/README.md (deflated 65%)
  adding: ecg_lora_adapter/tokenizer.json (deflated 81%)
  adding: ecg_lora_adapter/adapter_config.json (deflated 58%)
  adding: ecg_lora_adapter/adapter_model.safetensors (deflated 22%)

ZIP File Created: ecg_adapter.zip
